# 🚀 IndoClip Pro — Backend Runner (Kaggle GPU T4)

Notebook ini dirancang khusus untuk menjalankan backend **IndoClip Pro** di atas **Kaggle Notebooks** dengan akselerasi **GPU NVIDIA T4 (16 GB VRAM)**.

### ⚙️ Pengaturan Wajib Kaggle Sebelum Menjalankan:
1. **Accelerator:** Pastikan di panel kanan (Notebook Settings) Anda memilih **GPU T4 x2** (atau GPU P100).
2. **Internet:** Pastikan opsi **Internet** di panel kanan berada dalam posisi **ON (Active)**.
3. *(Opsional)* **Cookies YouTube:** Jika Anda memiliki file `cookies.txt`, cukup tambahkan sebagai dataset di Kaggle (`/kaggle/input/...`), notebook ini akan otomatis mendeteksi dan menyalinnya ke backend.

In [ ]:
# =============================================================================
# 🔑 1. KONFIGURASI API KEY & NGROK TUNNEL
# =============================================================================
# Masukkan token authtoken Ngrok dan API Key AI Anda di bawah ini:

NGROK_AUTHTOKEN = ""
NGROK_DOMAIN    = "brunette-crimson-yen.ngrok-free.dev"  # Domain statis Ngrok IndoClip

# API Key AI (Bisa diisi salah satu, otomatis Auto-Fallback):
GEMINI_API_KEY  = ""
MISTRAL_API_KEY = ""
GROQ_API_KEY    = ""

# =============================================================================
# 📦 2. INITIALIZE ENVIRONMENT & GIT REPOSITORY
# =============================================================================
import os, sys, shutil, glob, time, subprocess
from pathlib import Path

# Inject configuration to environment variables
if NGROK_AUTHTOKEN: os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
if NGROK_DOMAIN:    os.environ["NGROK_DOMAIN"]    = NGROK_DOMAIN
if GEMINI_API_KEY:  os.environ["GEMINI_API_KEY"]  = GEMINI_API_KEY
if MISTRAL_API_KEY: os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY
if GROQ_API_KEY:    os.environ["GROQ_API_KEY"]    = GROQ_API_KEY

# Set runtime GPU parameters for Faster-Whisper & FFmpeg
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
os.environ["WHISPER_MODEL"] = "small"
os.environ["FFMPEG_PRESET"] = "ultrafast"
os.environ["FFMPEG_THREADS"] = "0"

WORKDIR = Path("/kaggle/working/IndoClip")
REPO_URL = "https://github.com/nausamah27/IndoClip.git"

if not WORKDIR.exists():
    print("⏳ Meng-clone repositori IndoClip ke Kaggle...")
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
else:
    print("⏳ Memperbarui repositori IndoClip (git pull)...\n")
    subprocess.run(["git", "-C", str(WORKDIR), "pull"], check=True)

BACKEND_DIR = WORKDIR / "backend"
os.chdir(BACKEND_DIR)
print(f"📂 Direktori aktif: {os.getcwd()}")

# =============================================================================
# 🍪 3. AUTO-DETECT & COPY COOKIES.TXT (ANTI-BOT YOUTUBE)
# =============================================================================
print("🔍 Mencari file cookies.txt di /kaggle/input/...")
detected_cookies = list(glob.glob("/kaggle/input/**/cookies.txt", recursive=True))
target_cookie_file = BACKEND_DIR / "cookies.txt"

if detected_cookies:
    source_cookie = detected_cookies[0]
    shutil.copy(source_cookie, target_cookie_file)
    print(f"✅ cookies.txt berhasil disalin dari: {source_cookie}")
else:
    print("ℹ️  cookies.txt tidak ditemukan di /kaggle/input (opsional, proses tetap berjalan normal).")

# =============================================================================
# 🛠️ 4. INSTALL DEPENDENCIES & GPU LIBRARIES
# =============================================================================
print("\n⏳ Menginstall dependensi (hanya butuh ~1 menit)...\n")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libgl1", "libglib2.0-0"], check=False)

# Install python requirements
pip_install_cmd = [
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "pyngrok>=7.1.0",
    "faster-whisper",
    "yt-dlp",
    "google-genai",
    "opencv-python-headless",
    "mediapipe",
    "scipy",
    "pydantic",
    "fastapi",
    "uvicorn[standard]",
    "python-dotenv",
    "nvidia-cublas-cu12",
    "nvidia-cudnn-cu12"
]
subprocess.run(pip_install_cmd, check=True)

# Link NVIDIA cuBLAS 12 libraries ke LD_LIBRARY_PATH
import site
for p in site.getsitepackages():
    cublas_dir = os.path.join(p, "nvidia", "cublas", "lib")
    if os.path.isdir(cublas_dir):
        cur_ld = os.environ.get("LD_LIBRARY_PATH", "")
        if cublas_dir not in cur_ld:
            os.environ["LD_LIBRARY_PATH"] = f"{cublas_dir}:{cur_ld}"
            print(f"✅ Auto-configured cuBLAS library path: {cublas_dir}")

# =============================================================================
# 🚀 5. JALANKAN BACKEND INDOCLIP & NGROK TUNNEL
# =============================================================================
print("\n🚀 Menjalankan Backend IndoClip Pro di GPU Kaggle...")
runner_script = BACKEND_DIR / "run_ngrok.py"

if runner_script.exists():
    # Jalankan script run_ngrok.py bawaan repository
    subprocess.run([sys.executable, str(runner_script)], check=True)
else:
    # Fallback runner jika dipanggil dari luar
    from pyngrok import ngrok, conf
    if NGROK_AUTHTOKEN:
        try: ngrok.set_auth_token(NGROK_AUTHTOKEN)
        except Exception: pass
    tunnel = ngrok.connect(8000, "http", domain=NGROK_DOMAIN)
    print(f"\n🌟 IndoClip Backend Online: {tunnel.public_url}\n")
    subprocess.run([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"], check=True)
